# 11 — Genetic Interactions

**Prerequisites:** Run [09_mixscape.ipynb](09_mixscape.ipynb) first. Requires `data/norman_mixscape.h5ad`.

**What you'll learn:**
- The concept of genetic interaction (epistasis) in transcriptomics
- The additive null model: expected double-perturbation = sum of single effects
- How Norman et al. 2019 used dual-guide CRISPRa to map interaction landscapes
- Computing interaction scores from Perturb-seq data
- Classifying interactions as alleviating, aggravating, or non-interacting
- Comparing your results to the published Norman 2019 interaction map

**Dataset:** Norman 2019 — the only dataset in this course with systematic dual-guide perturbations, enabling interaction analysis.

**Estimated time:** 3 hours

In [ ]:
import os
import scanpy as sc
import pertpy as pt
import anndata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse
from scipy.stats import pearsonr
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")

sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=100, facecolor="white")

DATA_DIR = "data"
FIG_DIR = "figures"

adata = sc.read_h5ad(os.path.join(DATA_DIR, "norman_mixscape.h5ad"))
print(adata)

PERT_COL = "perturbation" if "perturbation" in adata.obs.columns else "condition"
ntc_mask = adata.obs["is_ntc"] if "is_ntc" in adata.obs.columns else \
    adata.obs[PERT_COL].str.lower().str.contains("ctrl|control|ntc|non")

print(f"Perturbations: {adata.obs[PERT_COL].nunique()}")

## 1. Concept: what is a genetic interaction?

When two genes A and B are perturbed together, there are three possible outcomes:

```
Additive (no interaction):
  effect(A+B) ≈ effect(A) + effect(B)

Alleviating (buffering / epistasis):
  effect(A+B) < effect(A) + effect(B)
  Interpretation: B doesn't add to A's effect — they're in the same pathway
  or B rescues A's effect

Aggravating (synergistic / synthetic lethal):
  effect(A+B) > effect(A) + effect(B)
  Interpretation: the two genes buffer each other; when both are gone,
  the effect is disproportionately large
```

In Perturb-seq, "effect" is a **transcriptome-wide vector**, not a single phenotype. The interaction score is thus a vector (one value per gene), and we summarize it by its magnitude or by projection onto relevant gene programs.

**Norman 2019 example:** CBFB and RUNX1 are subunits of the CBF transcription factor complex. When both are overexpressed (CRISPRa), the combined effect on the transcriptome is far smaller than the sum of individual effects — strong alleviating interaction, consistent with their co-regulation in the same complex.

## 2. Separate single and dual perturbation cells

In [ ]:
# Single-target cells (excluding NTC and dual)
is_dual = adata.obs["is_dual"] if "is_dual" in adata.obs.columns else \
    adata.obs[PERT_COL].str.contains("+", regex=False)

single_cells = adata[~is_dual & ~ntc_mask].copy()
dual_cells = adata[is_dual].copy()
ntc_cells = adata[ntc_mask].copy()

print(f"NTC cells: {len(ntc_cells)}")
print(f"Single-target cells: {len(single_cells)}, {single_cells.obs[PERT_COL].nunique()} perturbations")
print(f"Dual-target cells: {len(dual_cells)}, {dual_cells.obs[PERT_COL].nunique()} perturbation pairs")

# Show some dual-target perturbation labels
print("\nExample dual perturbations:")
print(sorted(dual_cells.obs[PERT_COL].unique())[:10])

In [ ]:
# Ensure normalized expression for signature computation
for adata_obj in [single_cells, dual_cells, ntc_cells]:
    X_s = adata_obj.X[:3, :3]
    if scipy.sparse.issparse(X_s):
        X_s = X_s.toarray()
    if np.allclose(X_s, np.round(X_s)):
        sc.pp.normalize_total(adata_obj, target_sum=1e4)
        sc.pp.log1p(adata_obj)

## 3. Compute single-perturbation effect profiles

In [ ]:
# NTC mean expression
X_ntc = ntc_cells.X
if scipy.sparse.issparse(X_ntc):
    X_ntc = X_ntc.toarray()
ntc_mean = X_ntc.mean(axis=0)
gene_names = ntc_cells.var_names.tolist()

# Per-gene single-perturbation signatures
single_signatures = {}  # gene -> np.array (perturbation signature vector)
single_cell_counts = {}  # gene -> n cells

for gene in single_cells.obs[PERT_COL].unique():
    cell_idx = single_cells.obs[PERT_COL] == gene
    X_gene = single_cells.X[cell_idx.values]
    if scipy.sparse.issparse(X_gene):
        X_gene = X_gene.toarray()
    
    # Use Mixscape-filtered cells if available
    if "mixscape_class" in single_cells.obs.columns:
        ko_mask_local = single_cells.obs.loc[cell_idx, "mixscape_class"] == "KO"
        if ko_mask_local.sum() >= 5:
            X_gene = X_gene[ko_mask_local.values]
    
    mean_expr = X_gene.mean(axis=0)
    signature = mean_expr - ntc_mean
    single_signatures[gene] = signature
    single_cell_counts[gene] = len(X_gene)

print(f"Single-perturbation signatures computed for {len(single_signatures)} genes")
print(f"Signature vector length (n genes): {len(gene_names)}")

## 4. Compute interaction scores

In [ ]:
def parse_dual_label(label, sep="+"):
    """Parse 'GENE_A+GENE_B' into ('GENE_A', 'GENE_B')."""
    parts = label.split(sep)
    if len(parts) == 2:
        return parts[0].strip(), parts[1].strip()
    return None, None


interaction_results = []

for pair_label in dual_cells.obs[PERT_COL].unique():
    gene_a, gene_b = parse_dual_label(pair_label)
    
    if gene_a is None or gene_b is None:
        continue
    if gene_a not in single_signatures or gene_b not in single_signatures:
        continue  # Need single-gene data for both
    
    # Observed double perturbation signature
    dual_idx = dual_cells.obs[PERT_COL] == pair_label
    X_dual = dual_cells.X[dual_idx.values]
    if scipy.sparse.issparse(X_dual):
        X_dual = X_dual.toarray()
    
    if len(X_dual) < 5:
        continue
    
    obs_signature = X_dual.mean(axis=0) - ntc_mean
    
    # Expected: additive model
    exp_signature = single_signatures[gene_a] + single_signatures[gene_b]
    
    # Interaction signature: observed - expected
    interaction_sig = obs_signature - exp_signature
    
    # Scalar interaction score: L2 norm of interaction signature
    # Sign: positive = aggravating (more than expected), negative = alleviating (less)
    # Use dot product with the expected direction as sign indicator
    interaction_magnitude = np.linalg.norm(interaction_sig)
    cos_sim_with_expected = np.dot(obs_signature, exp_signature) / (
        np.linalg.norm(obs_signature) * np.linalg.norm(exp_signature) + 1e-10
    )
    
    # Correlation between observed and expected (concordance measure)
    corr_obs_exp = np.corrcoef(obs_signature, exp_signature)[0, 1]
    
    interaction_results.append({
        "pair": pair_label,
        "gene_a": gene_a,
        "gene_b": gene_b,
        "n_cells": len(X_dual),
        "interaction_magnitude": interaction_magnitude,
        "corr_obs_exp": corr_obs_exp,
        "obs_magnitude": np.linalg.norm(obs_signature),
        "exp_magnitude": np.linalg.norm(exp_signature),
        "interaction_signature": interaction_sig,
    })

interaction_df = pd.DataFrame([{k: v for k, v in r.items() if k != "interaction_signature"}
                                for r in interaction_results])

print(f"Interaction scores computed for {len(interaction_df)} gene pairs")
print(interaction_df.head(10))

In [ ]:
# Classify interactions
# corr_obs_exp:
#   > 0.8: additive (observed matches expected)
#   0.3 - 0.8: weakly interacting
#   < 0.3: strong interaction

# obs_magnitude vs. exp_magnitude ratio:
#   ratio < 0.7: alleviating (observed is weaker than expected)
#   ratio > 1.3: aggravating (observed is stronger than expected)

interaction_df["magnitude_ratio"] = interaction_df["obs_magnitude"] / (interaction_df["exp_magnitude"] + 1e-10)

def classify_interaction(row):
    if row["corr_obs_exp"] > 0.8:
        return "additive"
    elif row["magnitude_ratio"] < 0.7:
        return "alleviating"
    elif row["magnitude_ratio"] > 1.3:
        return "aggravating"
    else:
        return "weak_interaction"

interaction_df["interaction_class"] = interaction_df.apply(classify_interaction, axis=1)

print("Interaction class distribution:")
print(interaction_df["interaction_class"].value_counts())

## 5. Observed vs. expected scatter

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: observed magnitude vs. expected magnitude
class_colors = {
    "additive": "#90A4AE",
    "alleviating": "#42A5F5",
    "aggravating": "#EF5350",
    "weak_interaction": "#FFA726",
}

for cls, color in class_colors.items():
    mask = interaction_df["interaction_class"] == cls
    axes[0].scatter(
        interaction_df.loc[mask, "exp_magnitude"],
        interaction_df.loc[mask, "obs_magnitude"],
        c=color, label=cls, alpha=0.7, s=40, edgecolors="none"
    )

max_val = max(interaction_df["exp_magnitude"].max(), interaction_df["obs_magnitude"].max())
axes[0].plot([0, max_val], [0, max_val], "k--", lw=1, label="additive")
axes[0].set_xlabel("Expected magnitude (sum of singles)")
axes[0].set_ylabel("Observed magnitude (double)")
axes[0].set_title("Observed vs. expected perturbation magnitude")
axes[0].legend()

# Label notable pairs
extreme_pairs = pd.concat([
    interaction_df.nsmallest(3, "magnitude_ratio"),
    interaction_df.nlargest(3, "magnitude_ratio")
])
for _, row in extreme_pairs.iterrows():
    axes[0].annotate(row["pair"].replace("+", "\n+"),
                     (row["exp_magnitude"], row["obs_magnitude"]),
                     fontsize=7, ha="center")

# Pie chart: interaction class distribution
class_counts = interaction_df["interaction_class"].value_counts()
axes[1].pie(
    class_counts,
    labels=class_counts.index,
    colors=[class_colors.get(c, "gray") for c in class_counts.index],
    autopct="%1.1f%%",
    startangle=90,
)
axes[1].set_title("Interaction class distribution")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "11_interactions_scatter.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Interaction heatmap

In [ ]:
# Build gene × gene interaction heatmap
# Use correlation between observed and expected as a signed interaction score
# Low correlation = strong interaction
# Sign: magnitude_ratio - 1 (positive = aggravating, negative = alleviating)

interaction_df["signed_score"] = (1 - interaction_df["corr_obs_exp"]) * np.sign(
    interaction_df["magnitude_ratio"] - 1
)

# Get all unique single genes that appear in dual perturbations
all_genes_in_pairs = sorted(set(
    interaction_df["gene_a"].tolist() + interaction_df["gene_b"].tolist()
))

n = len(all_genes_in_pairs)
gene_idx = {g: i for i, g in enumerate(all_genes_in_pairs)}
interaction_matrix = np.zeros((n, n))

for _, row in interaction_df.iterrows():
    i, j = gene_idx[row["gene_a"]], gene_idx[row["gene_b"]]
    interaction_matrix[i, j] = row["signed_score"]
    interaction_matrix[j, i] = row["signed_score"]  # symmetric

interact_df = pd.DataFrame(interaction_matrix, index=all_genes_in_pairs, columns=all_genes_in_pairs)

if n >= 4:
    g = sns.clustermap(
        interact_df,
        cmap="RdBu_r",
        center=0,
        vmin=-1,
        vmax=1,
        figsize=(max(8, n*0.4), max(6, n*0.4)),
        xticklabels=True,
        yticklabels=True,
        cbar_kws={"label": "Interaction score\n(+ aggravating, - alleviating)"},
    )
    g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=8, rotation=90)
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=8)
    g.fig.suptitle("Genetic interaction heatmap", y=1.01)
    g.savefig(os.path.join(FIG_DIR, "11_interaction_heatmap.png"), dpi=150, bbox_inches="tight")
    plt.show()

## 7. Compare to Norman 2019 published results

The Norman et al. 2019 paper includes a supplementary table with their interaction scores for all tested pairs. Notable strong interactions from the paper:

| Gene pair | Interaction type | Biological explanation |
|---|---|---|
| CBFB + RUNX1 | Alleviating | Both subunits of CBF complex — co-regulated |
| ETS2 + CEBPA | Alleviating | Both hematopoietic TFs — convergent targets |
| KLF1 + KLF4 | Alleviating | Paralogous KLF family members |
| FOSB + CEBPB | Aggravating | Cross-activation of distinct differentiation programs |

In [ ]:
# Check our scores for the canonical Norman pairs
canonical_pairs = [
    ("CBFB", "RUNX1"),
    ("ETS2", "CEBPA"),
    ("KLF1", "KLF4"),
    ("FOSB", "CEBPB"),
]

print("Canonical pair interaction scores from our analysis:")
print("-" * 60)

for gene_a, gene_b in canonical_pairs:
    # Check both orderings
    match = interaction_df[
        ((interaction_df["gene_a"] == gene_a) & (interaction_df["gene_b"] == gene_b)) |
        ((interaction_df["gene_a"] == gene_b) & (interaction_df["gene_b"] == gene_a))
    ]
    if len(match) > 0:
        row = match.iloc[0]
        print(f"{gene_a}+{gene_b}: class={row['interaction_class']:20s} "
              f"corr={row['corr_obs_exp']:.2f}  "
              f"ratio={row['magnitude_ratio']:.2f}  "
              f"n_cells={row['n_cells']}")
    else:
        print(f"{gene_a}+{gene_b}: NOT FOUND in dual perturbation cells")

In [ ]:
# Top interacting pairs
print("Top 10 most interacting pairs (by interaction magnitude):")
top_interactions = interaction_df.nlargest(10, "interaction_magnitude")[["pair", "interaction_class", "corr_obs_exp", "magnitude_ratio", "n_cells"]]
print(top_interactions.to_string(index=False))

print("\nTop 5 alleviating pairs:")
top_alleviating = interaction_df[interaction_df["interaction_class"] == "alleviating"].nsmallest(5, "magnitude_ratio")
print(top_alleviating[["pair", "magnitude_ratio", "corr_obs_exp", "n_cells"]].to_string(index=False))

print("\nTop 5 aggravating pairs:")
top_aggravating = interaction_df[interaction_df["interaction_class"] == "aggravating"].nlargest(5, "magnitude_ratio")
print(top_aggravating[["pair", "magnitude_ratio", "corr_obs_exp", "n_cells"]].to_string(index=False))

## Key takeaways

1. The additive null model is simple but powerful: `expected(A+B) = effect(A) + effect(B)`; deviations are the interaction signal
2. Alleviating interactions (observed < expected) suggest the two genes act in the same pathway or are co-regulated — one perturbation "uses up" the pathway's capacity
3. Aggravating interactions (observed > expected) suggest the two genes normally buffer each other — removing both is disproportionately disruptive
4. Interaction scores should be validated with concordant guides and checked for statistical significance (permutation test using NTC pairs)
5. The Norman 2019 dataset is the canonical benchmark for this analysis — cross-referencing your scores with the published supplementary table is a critical validation step

**Next:** [12_grn_and_prediction.ipynb](12_grn_and_prediction.ipynb)